# Task 4 — Join, Transformation Rules and Testing

Applies transformation rules to the validated freeCodeCamp dataset and saves the final analysis-ready CSV to `data/processed/final.csv`.

**Note on the "join" part:** this pipeline currently has a single source (freeCodeCamp), so there's no real join between two tables at this stage. Once other teammates' sources are merged into a shared `data/interim/validated.csv`, a join step would combine them here — see the join specification note below for what that would look like.

## Imports

In [16]:
import pandas as pd

## Load the validated data

In [17]:
df = pd.read_csv("../data/interim/validated.csv")
print(f"Loaded {len(df)} rows")
df.head()

Loaded 450 rows


,source,category,title,author,publication_date,description,url,topic,matched_keywords,publication_datetime
0,freeCodeCamp,#shadcn ui,How to Build an AI Chat App Interface With the...,Vaibhav Gupta,2026-09-11,Every other AI product you open today has the ...,https://www.freecodecamp.org/news/how-to-build...,AI,none,2026-09-11 16:21:29.864000+00:00
1,freeCodeCamp,#Artificial Intelligence,How to Build a Self-Evaluating AI System: Auto...,Jude Otine,2026-09-11,So you shipped your AI feature and it works in...,https://www.freecodecamp.org/news/build-a-self...,AI,"artificial intelligence, llm",2026-09-11 15:24:04.941000+00:00
2,freeCodeCamp,#Security,How AI Is Changing Malware Detection: From Tra...,Manish Shivanandhan,2026-09-11,Malware used to be simple to describe. A virus...,https://www.freecodecamp.org/news/how-ai-is-ch...,AI,none,2026-09-11 15:22:46.931000+00:00
3,freeCodeCamp,#AI,How to Build an AI Chatbot with Gemini and Ver...,Johnson Samuel,2026-09-07,"A couple of months back, I built a chatbot app...",https://www.freecodecamp.org/news/how-to-build...,AI,none,2026-09-07 22:35:39.060000+00:00
4,freeCodeCamp,#AI,How AI Receptionists Work: The Architecture Be...,Manish Shivanandhan,2026-09-04,An AI receptionist may sound simple from the o...,https://www.freecodecamp.org/news/how-ai-recep...,AI,none,2026-09-04 20:07:46.216000+00:00


## Join specification

**Current state:** single source, no join performed.

**Planned join (once merged with teammates' sources):**
- **Keys:** `url` (each article's URL is unique across all sources)
- **Join type:** not a traditional join — sources are *stacked* (concatenated) into one table, since each source contributes distinct rows rather than matching columns across tables
- **Expected row count:** sum of each source's validated row count (no overlap expected, since each source only scrapes its own site)
- **Duplicate handling:** deduplicate on `url` after concatenation, in case the same article is cross-posted on two sites

## Transformation rules

| Rule ID | Description | Input Column(s) | Output Column |
|---|---|---|---|
| R1 | Extract the publication year from the date | publication_date | publish_year |
| R2 | Count how many keywords matched | matched_keywords | keyword_count |
| R3 | Classify article length (short/medium/long) by description length | description | length_category |

## Implement the rules

In [18]:
df_final = df.copy()

# R1: publish_year from publication_date
df_final["publish_year"] = pd.to_datetime(df_final["publication_date"], errors="coerce").dt.year

# R2: keyword_count from matched_keywords
# matched_keywords is a comma-separated string, or "none" if empty
def count_keywords(value):
    if pd.isna(value) or str(value).strip().lower() == "none":
        return 0
    return len(str(value).split(","))

df_final["keyword_count"] = df_final["matched_keywords"].apply(count_keywords)

# R3: length_category from description length
def classify_length(description):
    if pd.isna(description):
        return "unknown"
    length = len(str(description))
    if length < 100:
        return "short"
    elif length < 200:
        return "medium"
    else:
        return "long"

df_final["length_category"] = df_final["description"].apply(classify_length)

df_final[["title", "publication_date", "publish_year", "matched_keywords",
          "keyword_count", "description", "length_category"]].head()

,title,publication_date,publish_year,matched_keywords,keyword_count,description,length_category
0,How to Build an AI Chat App Interface With the...,2026-09-11,2026,none,0,Every other AI product you open today has the ...,long
1,How to Build a Self-Evaluating AI System: Auto...,2026-09-11,2026,"artificial intelligence, llm",2,So you shipped your AI feature and it works in...,long
2,How AI Is Changing Malware Detection: From Tra...,2026-09-11,2026,none,0,Malware used to be simple to describe. A virus...,long
3,How to Build an AI Chatbot with Gemini and Ver...,2026-09-07,2026,none,0,"A couple of months back, I built a chatbot app...",long
4,How AI Receptionists Work: The Architecture Be...,2026-09-04,2026,none,0,An AI receptionist may sound simple from the o...,long


## Test each rule

Checks expected output, edge cases, empty input, and null keys — as required for this task.

In [19]:
print("Actual publish_year range:")
print(df_final["publish_year"].dropna().sort_values().unique())

assert df_final["publish_year"].dropna().between(2010, 2030).all(), "Unexpected publish_year values found"
assert pd.to_datetime("2026-09-11").year == 2026  # sanity check on the logic itself

# R2 tests
assert count_keywords("none") == 0, "R2 failed on empty case"
assert count_keywords(None) == 0, "R2 failed on null case"
assert count_keywords("ai, llm") == 2, "R2 failed on multi-keyword case"
assert count_keywords("ai") == 1, "R2 failed on single-keyword case"
assert (df_final["keyword_count"] >= 0).all(), "Negative keyword_count found"

# R3 tests
assert classify_length("") == "short", "R3 failed on empty string"
assert classify_length(None) == "unknown", "R3 failed on null case"
assert classify_length("x" * 50) == "short"
assert classify_length("x" * 150) == "medium"
assert classify_length("x" * 250) == "long"
assert df_final["length_category"].isin(["short", "medium", "long", "unknown"]).all()

print("All rule tests passed.")
print(f"\nlength_category distribution:\n{df_final['length_category'].value_counts()}")
print(f"\npublish_year distribution:\n{df_final['publish_year'].value_counts().sort_index()}")

Actual publish_year range:
[2016 2017 2018 2019 2020 2021 2022 2023 2024 2025 2026]
All rule tests passed.

length_category distribution:
length_category
long      445
medium      5
Name: count, dtype: int64

publish_year distribution:
publish_year
2016      2
2017     10
2018     15
2019     13
2020     18
2021     18
2022     14
2023     19
2024     73
2025     51
2026    217
Name: count, dtype: int64


## Save the final dataset

In [20]:
output_file = "../data/processed/final.csv"
df_final.to_csv(output_file, index=False)
print(f"Saved {len(df_final)} rows to {output_file}")

Saved 450 rows to ../data/processed/final.csv
